# Entrenamiento del modelo

--- 

Despues de la limpieza de datos y el preprocesamiento adecuado, seguiremos con el entrenamiento de diferentes modelos para obtener el mejor clasificador. Para el **Fine-tuning** usaremos `Optuna`, el registro de modelos se hara en `mlflow` hosteado en `databricks`. 

--- 

### Librerias:

In [13]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from dotenv import load_dotenv
import pandas as pd
import mlflow
import os
import optuna
import math
import pathlib
import pickle
from optuna.samplers import TPESampler
from sklearn.metrics import f1_score, precision_score, recall_score
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

Iniciamos el experimento y la conexión en databricks:

In [14]:
load_dotenv(override=True)
EXPERIMENT_NAME = "/Users/oscar.josue2204@gmail.com/income-prediction"

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

2025/11/11 19:11:11 INFO mlflow.tracking.fluent: Experiment with name '/Users/oscar.josue2204@gmail.com/income-prediction' does not exist. Creating a new experiment.


Leemos los datos

In [15]:
df = pd.read_csv("../data/raw/adult.csv")
df.head()

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


Creamos una clase que hace el preprocesamiento desarrollado en el notebook anterior:

In [16]:
def preprocessor(df):
    # Target binario
    y = df["income"].apply(lambda x: 1 if ">50K" in x else 0)

    # Features (quitamos income y education)
    X = df.drop(["income", "education"], axis=1)

    # Columnas categóricas y numéricas
    categorical_cols = [
        'workclass', 'marital.status', 
        'occupation', 'race', 'relationship', 
        'sex', 'native.country'
    ]
    numeric_cols = [col for col in X.columns if col not in categorical_cols]

    # Transformadores
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    # Pipeline de preprocesamiento
    preprocessor = ColumnTransformer([
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

    # Split inicial (train + temp)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    # Split secundario (val + test)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
    )

    # Ajustar y transformar solo con train
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)
    X_test_processed = preprocessor.transform(X_test)

    # Obtener nombres de columnas
    cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols)
    all_features = numeric_cols + cat_features.tolist()

    # Reconstruir DataFrames
    X_train_processed = pd.DataFrame(X_train_processed, columns=all_features, index=X_train.index)
    X_val_processed = pd.DataFrame(X_val_processed, columns=all_features, index=X_val.index)
    X_test_processed = pd.DataFrame(X_test_processed, columns=all_features, index=X_test.index)

    return X_train_processed, X_val_processed, X_test_processed, y_train, y_val, y_test

X_train_processed, X_val_processed, X_test_processed, y_train, y_val, y_test = preprocessor(df)


Ahora empezaremos con el entrenamiento de modelos:

---

## Random forest classifier

In [17]:


# ------------------------------------------------------------
# Definición del objetivo para Optuna
# ------------------------------------------------------------
def objective_rf(trial, X_train, y_train, X_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state": 42,
        "n_jobs": -1
    }

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "RandomForestClassifier")
        mlflow.log_params(params)

        model = RandomForestClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)

        mlflow.log_metrics({
            "f1": f1,
            "precision": precision,
            "recall": recall
        })

        
        input_example = X_val.head(5)
        signature = infer_signature(input_example, y_val.head(5))

        mlflow.sklearn.log_model(
            sk_model=model,
            artifact_path="model",
            input_example=input_example,
            signature=signature
        )

    # Queremos maximizar F1
    return 1 - f1



In [18]:

mlflow.sklearn.autolog(log_models=False)

sampler = TPESampler(seed=42)
study_rf = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="RandomForest Optuna Optimization"):
    study_rf.optimize(lambda trial: objective_rf(trial, X_train_processed, y_train, X_val_processed, y_val),
                      n_trials=10)

    best_params = study_rf.best_params
    best_params["random_state"] = 42
    mlflow.log_params(best_params)

    mlflow.set_tags({
        "project": "Adult Income Prediction",
        "optimizer_engine": "Optuna",
        "model_family": "RandomForestClassifier"
    })

    final_model = RandomForestClassifier(**best_params)
    final_model.fit(X_train_processed, y_train)

    y_pred = final_model.predict(X_test_processed)
    f1 = f1_score(y_test, y_pred)
    mlflow.log_metric("final_f1", f1)

    # Guardar preprocesador
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(preprocessor, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    # Registrar modelo final
    input_example = X_test_processed.head(5)
    signature = infer_signature(input_example, y_test.head(5))

    mlflow.sklearn.log_model(
        sk_model=final_model,
        artifact_path="model",
        input_example=input_example,
        signature=signature
    )


[I 2025-11-11 19:11:12,176] A new study created in memory with name: no-name-3ccc9231-4c18-4f9f-b6c9-9a5e395fbbb9
c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:11:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please 

🏃 View run placid-loon-551 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/aa73f25638cb4fac896b74cf86a5f041
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:12:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 19:12:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version.

🏃 View run useful-mole-748 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/19202687d1b4435d9f1464443dddf569
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:12:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 19:12:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version.

...

🏃 View run dashing-wasp-9 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/d1af93453b0c40d49094929be5f05088
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:16:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 19:16:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version.

🏃 View run unequaled-gnu-630 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/78c154b2ab934ae8b11030e2123d2a8e
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:16:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 19:17:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version.

🏃 View run RandomForest Optuna Optimization at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/dd5532deb86d452483c2c57c2ff82609
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


## Gradient Boost Classifier

In [19]:
def objective_gb(trial, X_train, y_train, X_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 10),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "random_state": 42
    }

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "GradientBoostingClassifier")
        mlflow.log_params(params)

        model = GradientBoostingClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)

        mlflow.log_metrics({"f1": f1, "precision": precision, "recall": recall})

        input_example = X_val.head(5)
        signature = infer_signature(input_example, y_val.head(5))
        mlflow.sklearn.log_model(model, "model", input_example=input_example, signature=signature)

    return 1 - f1
    


In [20]:
mlflow.sklearn.autolog(log_models=False)
sampler = TPESampler(seed=42)
study_gb = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="GradientBoosting Optuna Optimization"):
    study_gb.optimize(lambda trial: objective_gb(trial, X_train_processed, y_train, X_val_processed, y_val),
                      n_trials=10)

    best_params = study_gb.best_params
    best_params["random_state"] = 42
    mlflow.log_params(best_params)

    mlflow.set_tags({
        "project": "Adult Income Prediction",
        "optimizer_engine": "Optuna",
        "model_family": "GradientBoostingClassifier"
    })

    final_model = GradientBoostingClassifier(**best_params)
    final_model.fit(X_train_processed, y_train)

    y_pred = final_model.predict(X_test_processed)
    f1 = f1_score(y_test, y_pred)
    mlflow.log_metric("final_f1", f1)

    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(preprocessor, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    input_example = X_test_processed.head(5)
    signature = infer_signature(input_example, y_test.head(5))
    mlflow.sklearn.log_model(final_model, "model", input_example=input_example, signature=signature)


[I 2025-11-11 19:17:11,662] A new study created in memory with name: no-name-eed10901-61df-4591-9983-0489e82595a4
c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:17:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please 

🏃 View run handsome-snipe-351 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/1158b7258ea5403eb12871be2eba3e18
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:18:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 19:18:26 WARNING mlflow.utils.environment: Failed to resolve installed pip version.

🏃 View run capricious-moose-446 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/e2d4507f7f234278beefc0530fe1c52e
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:18:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 19:19:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version.

...

🏃 View run stylish-bat-440 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/3a0af730fac645adab7975b680d7a6c5
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:23:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 19:24:01 WARNING mlflow.utils.environment: Failed to resolve installed pip version.

🏃 View run able-shrimp-189 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/2d9eb42ef20d4e1a8ce407ee75bf43b7
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:24:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 19:25:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version.

🏃 View run GradientBoosting Optuna Optimization at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/c6434a5122e648bd8216c2ef182d6ddb
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


## XBG Classifier

In [21]:


def objective_xgb(trial, X_train, y_train, X_val, y_val):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 400),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "use_label_encoder": False,
        "eval_metric": "logloss"
    }

    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "XGBClassifier")
        mlflow.log_params(params)

        model = XGBClassifier(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_pred)
        precision = precision_score(y_val, y_pred)
        recall = recall_score(y_val, y_pred)

        mlflow.log_metrics({"f1": f1, "precision": precision, "recall": recall})

        input_example = X_val.head(5)
        signature = infer_signature(input_example, y_val.head(5))
        mlflow.sklearn.log_model(model, "model", input_example=input_example, signature=signature)

    return 1 - f1



In [22]:

mlflow.sklearn.autolog(log_models=False)
sampler = TPESampler(seed=42)
study_xgb = optuna.create_study(direction="minimize", sampler=sampler)

with mlflow.start_run(run_name="XGBoost Optuna Optimization"):
    study_xgb.optimize(lambda trial: objective_xgb(trial, X_train_processed, y_train, X_val_processed, y_val),
                       n_trials=10)

    best_params = study_xgb.best_params
    best_params["random_state"] = 42
    mlflow.log_params(best_params)

    mlflow.set_tags({
        "project": "Adult Income Prediction",
        "optimizer_engine": "Optuna",
        "model_family": "XGBClassifier"
    })

    final_model = XGBClassifier(**best_params)
    final_model.fit(X_train_processed, y_train)

    y_pred = final_model.predict(X_test_processed)
    f1 = f1_score(y_test, y_pred)
    mlflow.log_metric("final_f1", f1)

    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(preprocessor, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    input_example = X_test_processed.head(5)
    signature = infer_signature(input_example, y_test.head(5))
    mlflow.sklearn.log_model(final_model, "model", input_example=input_example, signature=signature)


[I 2025-11-11 19:25:20,147] A new study created in memory with name: no-name-3b90e7e7-361a-4b28-bdb1-2926044b3127
c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:25:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) w

🏃 View run smiling-swan-838 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/911df475d808437f856b394064171951
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:25:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/

🏃 View run charming-lark-797 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/c19d1eb836c54f2ea45d32f252ceebe6
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:25:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/

...

🏃 View run upbeat-pug-79 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/2589248f79724f6790e4a4a9c14f1331
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [19:27:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/

🏃 View run debonair-cow-832 at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/9ab4bc63ee6a474fa40fc45799565c45
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


c:\Users\Rafa\apps\5to semestre\CdD\Data_Science_Final_Project\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2025/11/11 19:27:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/11 19:27:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version.

🏃 View run XGBoost Optuna Optimization at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275/runs/3899b8909d644e62ac4c39d326142a37
🧪 View experiment at: dbc-f2fdebc8-23c1.cloud.databricks.com/ml/experiments/4164948439788275


In [23]:
model_name = "workspace.default.income-prediction-classifier"

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.f1 ASC"],
    output_format="list"
)

best_run = runs[0]

result = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name=model_name
)

Successfully registered model 'workspace.default.income-prediction-classifier'.
2025/11/11 19:27:35 WARNING mlflow.tracking._model_registry.fluent: Run with id 28ebdd26972c4202b6a4b79af4961b78 has no artifacts at artifact path 'model', registering model based on models:/m-42bb6b308f0a4f8bab9f525b19f140dc instead
Uploading artifacts: 100%|██████████| 9/9 [00:04<00:00,  2.20it/s]
Created version '1' of model 'workspace.default.income-prediction-classifier'.


Register the champion:

In [24]:
from mlflow import MlflowClient

client = MlflowClient()

In [25]:
model_version = result.version
new_alias = "Champion"

client.set_registered_model_alias(
    name=model_name,
    alias=new_alias,
    version=result.version
)